### Download/ Querry Data from Jarvis NIST 

In [39]:
import requests
import pandas as pd
import json
import re
import numpy as np 

def lattice_vectors_to_parameters(lattice_vectors):
    """
    Calculation of the lattice parameters (a, b, c, alpha, beta, gamma) out of three lattice vectors.

    Args:
        lattice_vectors: list of three vectors [[ax, ay, az], [bx, by, bz], [cx, cy, cz]]

    Returns:
        dict with a, b, c, alpha, beta, gamma
    """
    a_vec = np.array(lattice_vectors[0])
    b_vec = np.array(lattice_vectors[1])
    c_vec = np.array(lattice_vectors[2])

    # length
    a = np.linalg.norm(a_vec)
    b = np.linalg.norm(b_vec)
    c = np.linalg.norm(c_vec)

    # degree 
    alpha = np.degrees(np.arccos(np.dot(b_vec, c_vec) / (b * c)))
    beta  = np.degrees(np.arccos(np.dot(a_vec, c_vec) / (a * c)))
    gamma = np.degrees(np.arccos(np.dot(a_vec, b_vec) / (a * b)))

    return {
        "a": a,
        "b": b,
        "c": c,
        "alpha": alpha,
        "beta": beta,
        "gamma": gamma
    }
# Query 
def jarvis_optimade_query(query):
    base_url = "https://jarvis.nist.gov/optimade/jarvisdft/v1/structures/?filter="
    url = base_url + query
    results = []

    while url:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        results.extend(data["data"])
        url = data.get("links", {}).get("next")

    return results


# Filter for elements (Refractory Elements) 
element_list = ["W", "Mo", "Ta", "Nb", "V", "Cr", "Hf", "Zr", "Ti", "Re"]
#refractory_elements = ["W"]
entries = []

# scanning trough all elements in refractory_elements
for el in element_list:
    query = f"elements HAS ANY {el}"  # containing the el from refractory elements 
    #query = "elements HAS ANY " + ",".join(el)
    print("Scanning:", el)
    results = jarvis_optimade_query(query)
    entries.extend(results)  # extend statt append!

#Filter for unique entries
unique_entries = {e["id"]: e for e in entries}
entries = list(unique_entries.values())
print("Found number of unique materials:", len(entries))



# Filter for local properties
materials_data = []

for entry in entries:
    attr = entry["attributes"]
    
    # Basic-Data
    mat_dict = {
        "material_id": entry["id"],
        "Chemical Formula": attr.get("chemical_formula_reduced"),
        "Chemical Formula Anonymus": attr.get("chemical_formula_anonymous"),
        "Chemical Formula Descriptive": attr.get("chemical_formula_descriptive"),
        "Bulk_modulus Voigt (GPa)": attr.get("_jarvis_bulk_modulus_kv"),
        "Shear modulus Voigt (GPa)": attr.get("_jarvis_shear_modulus_gv"),
        "Poisson ratio": attr.get("_jarvis_poisson"),
        "Crystal System": attr.get("_jarvis_crys"),
        "Spacegroup Symbol": attr.get("_jarvis_spg_symbol"),
        "Density (g/cm3)": attr.get("_jarvis_density"),
        "n_elements": attr.get("nelements"),
        "Material Type": attr.get("_jarvis_typ"),
        "Band Gap": attr.get("_jarvis_mbj_bandgap"),
        "Dimensionality": attr.get("_jarvis_dimensionality") 
    }

    # Calculation of lattice vectors and adding them to the dictionary 
    lattice_vectors = attr.get("lattice_vectors")
    if lattice_vectors:
        mat_dict.update(lattice_vectors_to_parameters(lattice_vectors))
    
    materials_data.append(mat_dict)

# Dataframe properties 
df = pd.DataFrame(materials_data)
print(df.shape) 
print(df.head())

base_folder = "Jarvis_Data"
os.makedirs(base_folder, exist_ok=True)
name = "Refractory_Metals_Data" # File name 
safe_name = re.sub(r'[<>:"/\\|?*]', "_", name)
csv_filepath = os.path.join(base_folder, f"{safe_name}.csv")
df.to_csv(csv_filepath, index=False)

print("Done. Dataframe contains all formulas and their properties.")

Scanning: W
Scanning: Mo
Scanning: Ta
Scanning: Nb
Scanning: V
Scanning: Cr
Scanning: Hf
Scanning: Zr
Scanning: Ti
Scanning: Re
Gefundene eindeutige Materialien: 8126
(8126, 20)
           material_id Chemical Formula Chemical Formula Anonymus  \
0   dft_2d_JVASP-60433            O4WZn                      A4BC   
1  dft_3d_JVASP-100056            F6WZn                      A6BC   
2   dft_3d_JVASP-10040         CrO6WZn2                    A6B2CD   
3  dft_3d_JVASP-100564        BaO6SrWZn                    A6BCDE   
4  dft_3d_JVASP-106282           Ru2WZn                      A2BC   

  Chemical Formula Descriptive  Bulk_modulus Voigt (GPa)  \
0                        ZnWO4                 -99999.00   
1                        ZnWF6                     38.40   
2                     Zn2CrWO6                 -99999.00   
3                    BaSrZnWO6                 -99999.00   
4                       ZnRu2W                    261.57   

   Shear modulus Voigt (GPa)  Poisson ratio Cr

In [17]:
import requests

# Elemente definieren (Refractory Metals)
refractory_elements = ["W"]

# OPTIMADE Query: wir fragen nur nach Elementen
query = "elements HAS ANY " + ",".join(refractory_elements)  # "HAS ANY" für mindestens ein Element
base_url = "https://jarvis.nist.gov/optimade/jarvisdft/v1/structures/?filter="

url = base_url + query

material_ids = []

# API Pagination
while url:
    r = requests.get(url)
    r.raise_for_status()
    data = r.json()
    
    # Alle Material-IDs sammeln
    for entry in data["data"]:
        material_ids.append(entry["id"])
    
    # Nächste Seite
    url = data.get("links", {}).get("next")

print(f"Anzahl gefundener Materialien: {len(material_ids)}")
print("Beispiel Material-IDs:", material_ids[:10])

materials = []

for mid in material_ids:
    url = f"{base_url}{mid}"
    r = requests.get(url)
    if r.ok:
        data = r.json()["data"]  # data ist eine Liste
        if len(data) > 0:
            attr = data[0]["attributes"]  # Zugriff auf das erste Element
            materials.append({
                "material_id": mid,
                "formula": attr.get("chemical_formula_reduced")
            })
    else:
        print(f"Fehler bei {mid}")

# DataFrame erstellen
df = pd.DataFrame(materials)
print(df.head())

Anzahl gefundener Materialien: 107
Beispiel Material-IDs: ['dft_2d_JVASP-60433', 'dft_3d_JVASP-100056', 'dft_3d_JVASP-10040', 'dft_3d_JVASP-100564', 'dft_3d_JVASP-106282', 'dft_3d_JVASP-109082', 'dft_3d_JVASP-110114', 'dft_3d_JVASP-110946', 'dft_3d_JVASP-111760', 'dft_3d_JVASP-113705']
Empty DataFrame
Columns: []
Index: []


In [23]:
import requests
import json

query = "elements HAS ANY W"
url = f"https://jarvis.nist.gov/optimade/jarvisdft/v1/structures/?filter={query}&page_limit=5"

r = requests.get(url)
r.raise_for_status()

data = r.json().get("data", [])
print(f"Gefundene Materialien: {len(data)}")

if len(data) > 0:
    entry = data[0] # erstes Material
    print(json.dumps(entry, indent=4))  # schön formatiert ausgeben

Gefundene Materialien: 20
{
    "id": "dft_2d_JVASP-60433",
    "type": "structures",
    "attributes": {
        "nsites": 6,
        "search": "O-W-Zn",
        "species": [
            {
                "mass": null,
                "name": "O",
                "attached": null,
                "nattached": null,
                "concentration": [
                    1.0
                ],
                "original_name": null,
                "chemical_symbols": [
                    "O"
                ]
            },
            {
                "mass": null,
                "name": "W",
                "attached": null,
                "nattached": null,
                "concentration": [
                    1.0
                ],
                "original_name": null,
                "chemical_symbols": [
                    "W"
                ]
            },
            {
                "mass": null,
                "name": "Zn",
                "attached": null,
        